# ESI HV ADC probe (discharge question)

The runtime's verified shutdown has never confirmed HV discharge on the real ESI. After it re-selects an ADC channel, it reads hundreds of volts (1324 V on HV1, 110 V on HV2 on 2026-10-05), although routine readings on the same channel stayed near 0 V for hours and HV was never enabled. This probe records what the ADC returns after each channel selection, to tell a settling or measurement artifact from a persistent reading. **It does not change the discharge criterion and does not certify discharge.**

## What it does

1. Connects through the bundled runtime, which forces HV and heater OFF (zero HV targets, modules disabled, global gate off).
2. Refuses to continue unless both HV modules read back inactive with a zero target and the heater module is inactive.
3. **Baseline:** `BASELINE_CONVERSIONS` fresh conversions on each module's untouched ADC channel.
4. **Channel selections:** `REPEATS` times, select the positive then the negative channel of each module and record the value available at once, then `CONVERSIONS` fresh conversions. A conversion is fresh when its ready flag was seen clear, then set: the same handshake as the runtime's check. Each row keeps the raw ready/overflow flags and the time since selection.
5. Restores the original channels, then runs the runtime's normal verified shutdown, including its own discharge check, recorded round by round.

## Safety

- Only the ADC measurement channel is changed, as the runtime's own discharge check does. No output, gate, target or heater is enabled.
- Keep the interlock available and verify HV OFF independently. **Treat the outputs as live until measured**: this notebook reads the controller's ADC, it is not a voltmeter on the outputs.
- If possible, measure HV1 and HV2 with an independent HV probe while the notebook runs, and note the readings with their times. Only that measurement can confirm or exclude a real voltage.

## Before running

Close Explorer, CGC utilities and every former ESI-owning kernel. Use a fresh 64-bit Windows kernel, with `notebooks/` beside `esi/` (or set `PLUGINS_DIR`). Check `ESI_COM`. The probe takes a few minutes.

The notebook uses the same per-COM guard as the heater and pressure–temperature notebooks. After an unfinished run, it asks for the run-specific `RESTART …` declaration. While the discharge question is open, the probe will normally end with the runtime's discharge check unconfirmed: the guard is then retained, and the next run of any ESI notebook asks for the declaration again.

**Stop:** interrupt the kernel once. The probe finishes the current conversion, restores the ADC channels, then performs the verified shutdown. Do not restart the kernel while shutdown is pending.

## Results

Saved under `esi_adc_probe_runs/<run>/`:

- `samples.csv`: one row per conversion (phase, module, channel, index, time since selection, flags, voltage, current, validity);
- `metadata.json`: settings, initial diagnostics, original channels, per-sequence summary, hints, and every round of the runtime's discharge check (`discharge_observations`);
- `native_calls.jsonl`: every native DLL call with its arguments, status and result;
- `adc_probe.png`: |voltage| versus time since channel selection for each module.

Each channel sequence is summarized as **low** (every fresh conversion within 1 V), **settles** (high at first, then within 1 V for the last five conversions), **stays high**, or **no fresh conversion**. These are hints for the next step, not conclusions.

In [ ]:
# Close Explorer, CGC utilities and every former ESI-owning kernel first.
# Only the HV ADC measurement channel is changed, as the runtime's own discharge check does;
# no output, gate, target or heater is ever enabled.
PLUGINS_DIR = None  # Optional: folder containing esi/; otherwise found nearby or in Explorer's plugin folder.
ESI_COM = 16
ESI_BAUD = 230400
REPEATS = 3  # Positive then negative channel selections per HV module.
CONVERSIONS = 20  # Fresh conversions recorded after each channel selection.
BASELINE_CONVERSIONS = 20  # Fresh conversions on the untouched original channel, before any selection.
CONVERSION_TIMEOUT_S = 3.0  # Longest wait for one fresh conversion; a stuck ready flag ends that sequence.
POLL_S = 0.01  # Ready-flag polling period.
LIMIT_V = 1.0  # The runtime's discharge threshold; used here only to summarize.

import csv
import functools
import hashlib
import importlib.machinery
import importlib.util
import json
import math
import os
import queue
import statistics
import sys
import threading
import time
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4

NOTEBOOK_DIR = Path(globals().get("__file__", Path.cwd() / "esi_adc_probe.ipynb")).resolve().parent
OUTPUT_DIR = NOTEBOOK_DIR / "esi_adc_probe_runs"
# Retain an uncertain transport across cell re-execution; restarting a kernel is NOT an OFF.
_ADC_GUARD = globals().get("_ADC_GUARD", {})

HV_MODULES = (1, 2)
# COM-ESI-CTRL.h: COM_ESI_CTRL_HVCTRL_ADCV_RDY/OFL and ADCI_RDY/OFL module data-ready flags.
READY_V, OVERFLOW_V, READY_I, OVERFLOW_I = 1 << 4, 1 << 5, 1 << 6, 1 << 7
READY = READY_V | READY_I
FIELDS = ("phase", "repeat", "module", "negative_channel", "high_current_range", "index", "fresh",
          "since_selection_s", "elapsed_s", "utc", "flags", "v_ready", "v_overflow", "i_ready", "i_overflow",
          "voltage_valid", "voltage_v", "current_valid", "current_a", "note")


def utc(timestamp=None):
    return datetime.fromtimestamp(time.time() if timestamp is None else timestamp, timezone.utc).isoformat()


def json_safe(value):
    if isinstance(value, float) and not math.isfinite(value):
        return None
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    return value


def sync_directory(path):
    if os.name != "nt":
        descriptor = os.open(path, os.O_RDONLY | os.O_DIRECTORY)
        try:
            os.fsync(descriptor)
        finally:
            os.close(descriptor)


def save_json(path, data):
    temporary = path.with_suffix(".tmp")
    with temporary.open("w", encoding="utf-8") as stream:
        json.dump(json_safe(data), stream, indent=2, allow_nan=False)
        stream.flush()
        os.fsync(stream.fileno())
    temporary.replace(path)
    sync_directory(path.parent)


def explorer_plugin_dirs():
    """ESIBD Explorer's configured plugin folder (Windows settings), then its default."""
    found = []
    try:
        import winreg
        with winreg.OpenKey(winreg.HKEY_CURRENT_USER, r"Software\ESIBD LAB\ESIBD Explorer\General") as key:
            value = winreg.QueryValueEx(key, "Plugin path")[0]
        if isinstance(value, str) and value and not value.startswith("@"):
            found.append(Path(value).expanduser())
    except (ImportError, OSError):
        pass
    found.append(Path.home() / "ESIBD Explorer" / "plugins")
    return found


def find_plugins(configured=None, notebook_dir=None):
    start = Path(notebook_dir or NOTEBOOK_DIR).resolve()
    candidates = [Path(configured).expanduser().resolve()] if configured is not None else [
        candidate for parent in (start, *start.parents) for candidate in (parent, parent / "plugins")
    ] + explorer_plugin_dirs()
    for root in candidates:
        if (root / "esi/vendor/runtime/__init__.py").is_file() and (root / "esi/_experiment_guard.py").is_file():
            return root
    raise FileNotFoundError("esi/ not found beside the notebook nor in ESIBD Explorer's plugin folder ("
                            + ", ".join(map(str, explorer_plugin_dirs())) + "). Set PLUGINS_DIR once.")


def load_private(path, package=False, fresh=False):
    path = Path(path).resolve()
    digest = hashlib.sha256(os.path.normcase(str(path)).encode()).hexdigest()[:16]
    prefix = "_esibd_adc_probe_helper_" if path.name == "_experiment_guard.py" else "_esibd_bundled_adc_probe_"
    name = prefix + digest
    if fresh and any(key == name or key.startswith(name + ".") for key in sys.modules):
        raise RuntimeError("ESI runtime already loaded. After confirmed hardware OFF, restart the kernel; never bypass an unfinished-run guard.")
    sources = sorted(path.parent.rglob("*.py")) if package else [path]
    fingerprint = {os.path.normcase(str(p.resolve())): hashlib.sha256(p.read_bytes()).hexdigest() for p in sources}
    if name in sys.modules and getattr(sys.modules[name], "_source_fingerprint", None) != fingerprint:
        raise RuntimeError("Bundled runtime changed after import. Confirm hardware OFF, then restart the kernel.")
    if name not in sys.modules:
        class SourceLoader(importlib.machinery.SourceFileLoader):
            def get_code(self, fullname):
                source = self.get_data(self.path)
                if hashlib.sha256(source).hexdigest() != fingerprint.get(os.path.normcase(str(Path(self.path).resolve()))):
                    raise RuntimeError("Runtime changed during import; restart the kernel.")
                return self.source_to_code(source, self.path)  # Compile verified bytes, never a stale .pyc.

        def source_spec(fullname, file, is_package):
            return importlib.util.spec_from_file_location(
                fullname, file, loader=SourceLoader(fullname, str(file)),
                submodule_search_locations=[str(file.parent)] if is_package else None)

        class PrivateFinder:
            def find_spec(self, fullname, path=None, target=None):
                if not fullname.startswith(name + "."):
                    return None
                relative = fullname[len(name) + 1:].split(".")
                file = package_root.joinpath(*relative)
                is_package = file.is_dir()
                file = file / "__init__.py" if is_package else file.with_suffix(".py")
                return source_spec(fullname, file, is_package) if file.is_file() else None

        package_root = path.parent
        finder = PrivateFinder()
        spec = source_spec(name, path, package)
        if spec is None or spec.loader is None:
            raise ModuleNotFoundError(str(path))
        module = importlib.util.module_from_spec(spec)
        sys.modules[name] = module
        try:
            sys.meta_path.insert(0, finder)  # Restricted to this new private namespace; sys.path is untouched.
            spec.loader.exec_module(module)
            if fingerprint != {os.path.normcase(str(p.resolve())): hashlib.sha256(p.read_bytes()).hexdigest() for p in sources}:
                raise RuntimeError("Runtime changed during import; restart the kernel.")
            module._source_fingerprint = fingerprint
        except BaseException:
            # Keep a failed-load marker: do not reuse partially imported package children.
            module._source_fingerprint = None
            raise
        finally:
            if finder in sys.meta_path:
                sys.meta_path.remove(finder)
    return sys.modules[name]


# Recorded with each run (provenance only; any plugin version may be used).
_RUNTIME_FILES = (
    "esi/vendor/runtime/esi/esi.py",
    "esi/vendor/runtime/esi/esi_base.py",
    "esi/vendor/runtime/_driver_common.py",
    "esi/vendor/runtime/esi/vendor/x64/COM-ESI-CTRL.dll",
    "esi/_experiment_guard.py",
)


def runtime_fingerprints(root):
    """SHA-256 of the runtime files used by this run, for its metadata only."""
    return {path: hashlib.sha256((root / path).read_bytes()).hexdigest()
            for path in _RUNTIME_FILES if (root / path).is_file()}


_DISCHARGE_RUNTIME_REQUIRED = "The loaded ESI runtime lacks the verified HV discharge check; update the esi/ plugin."


def require_discharge_runtime(runtime):
    # Check the class actually imported: same modules, same ready bits as this probe.
    try:
        controller = runtime.ESI._PROCESS_CONTROLLER_CLASS
        valid = (controller.__module__.startswith(runtime.__name__ + ".")
                 and callable(getattr(controller, "_verify_hv_discharge", None))
                 and tuple(controller.HV_MODULE_ADDRESSES) == HV_MODULES
                 and (controller.HV_ADC_V_READY, controller.HV_ADC_I_READY) == (READY_V, READY_I))
    except (AttributeError, TypeError) as error:
        raise RuntimeError(_DISCHARGE_RUNTIME_REQUIRED) from error
    if not valid:
        raise RuntimeError(_DISCHARGE_RUNTIME_REQUIRED)


def inline_backend(esi, base):
    """The runtime's in-process controller, whose lock and timeout every probe call uses."""
    try:
        backend = object.__getattribute__(esi, "_backend")
    except AttributeError:
        backend = esi
    if not isinstance(backend, base):
        raise RuntimeError("The ADC probe needs the inline ESI backend (process_backend=False); no connection attempted.")
    return backend


def hv_off_problem(diagnostic):
    for address in HV_MODULES:
        module = diagnostic["modules"][address]
        if module["module_active"] or module["active"] or module["target_v"] != 0:
            return f"HV module {address} is not confirmed OFF with zero target; probe refused"
    if diagnostic["heat"]["module_active"]:
        return "Heater module is not confirmed OFF; probe refused"
    return None


class NativeTrace:
    """Observe existing base-wrapper calls only; disk I/O stays off the DLL thread."""
    METHODS = ('open_port', 'close_port', 'get_heat_ctrl_monitoring',
               'get_complete_state', 'get_module_data_ready_flags',
               'get_hv_supply_meas_ranges', 'set_hv_supply_meas_ranges',
               'get_hv_supply_output_voltage', 'get_hv_supply_output_current',
               'get_hv_supply_target_output_voltage', 'set_hv_supply_target_output_voltage',
               'get_heat_ctrl_heater_temperature', 'set_heat_ctrl_heater_temperature',
               'get_module_activation_state', 'set_module_activation_state', 'get_enable', 'set_enable')

    def __init__(self, directory):
        self.path = Path(directory) / 'native_calls.jsonl'
        self.stream = self.path.open('x', encoding='utf-8')
        self.queue = queue.SimpleQueue()
        self.errors, self.hooks = [], []
        self.closed = False
        self.call_ids = iter(range(sys.maxsize))
        self.thread = threading.Thread(target=self._write, name='esi-adc-probe-trace-writer', daemon=True)
        self.thread.start()

    @staticmethod
    def encode(value):
        # Preserve non-finite values distinctly; unlike plotting JSON, never turn them into null.
        if isinstance(value, float) and not math.isfinite(value):
            return {'nonfinite_float': repr(value)}
        if isinstance(value, dict):
            return {str(k): NativeTrace.encode(v) for k, v in value.items()}
        if isinstance(value, (tuple, list)):
            return [NativeTrace.encode(v) for v in value]
        return value

    def record(self, event, **fields):
        try:
            if self.closed:
                return
            self.queue.put(dict(event=event, monotonic_ns=time.monotonic_ns(),
                                utc_ns=time.time_ns(), thread_id=threading.get_ident(),
                                **self.encode(fields)))
        except BaseException as error:
            if len(self.errors) < 10:
                self.errors.append(type(error).__name__)  # Never substitute a logging error for a native result.

    def _write(self):
        try:
            while True:
                row = self.queue.get()
                if row is None:
                    break
                try:
                    self.stream.write(json.dumps(self.encode(row), allow_nan=False) + '\n')
                    self.stream.flush()
                except BaseException as error:
                    if len(self.errors) < 10:
                        self.errors.append(f'{type(error).__name__}: {error}')
        finally:
            try:
                self.stream.close()
            except BaseException as error:
                self.errors.append(type(error).__name__)

    def install(self, base, backend):
        originals = [(name, getattr(base, name)) for name in self.METHODS]
        if not isinstance(backend, base) or not all(callable(f) for _, f in originals):
            raise RuntimeError('Native tracing requires the matched inline ESI backend; no connection attempted.')
        def wrap(name, original):
            def observed(device, *args, **kwargs):
                if device is not backend:
                    return original(device, *args, **kwargs)
                call_id = next(self.call_ids)
                self.record('native_start', call_id=call_id, method=name, args=args, kwargs=kwargs)
                try:
                    result = original(device, *args, **kwargs)
                except BaseException as error:
                    self.record('native_exception', call_id=call_id, method=name,
                                exception_type=type(error).__name__, message=str(error))
                    raise
                self.record('native_return', call_id=call_id, method=name, result=result)
                return result  # Same object/status/Valid, including false and non-finite data.
            return functools.wraps(original)(observed)
        for name, original in originals:
            observed = wrap(name, original)
            self.hooks.append((base, name, original, observed))
            setattr(base, name, observed)

    def finish(self, native_finished):
        if native_finished and not self.closed:
            for base, name, original, observed in self.hooks:
                if getattr(base, name) is observed:
                    setattr(base, name, original)
            self.closed = True
            self.queue.put(None)
            self.thread.join()
        # On poison/unknown state keep hooks/writer alive for a possible late native return.
        return {'file': self.path.name, 'writer_closed': self.closed,
                'native_instrumented': bool(self.hooks), 'errors': list(self.errors),
                'late_return_possible': not native_finished}


def summarize(rows, limit=LIMIT_V):
    """One entry per channel sequence; the pattern describes readings, it does not certify discharge."""
    groups = {}
    for row in rows:
        groups.setdefault((row["phase"], row["repeat"], row["module"], row["negative_channel"]), []).append(row)
    summary = []
    for (phase, repeat, module, negative), group in groups.items():
        def volts(row):
            value = row["voltage_v"]
            return float(value) if row["voltage_valid"] is True and math.isfinite(value) else math.nan
        immediate = next((volts(row) for row in group if row["index"] == 0), math.nan)
        fresh = [row for row in group if row["fresh"]]
        values = [volts(row) for row in fresh]
        high = [i for i, value in enumerate(values) if not abs(value) <= limit]  # Invalid counts as not low.
        tail = values[-5:]
        if not fresh:
            pattern = "no fresh conversion"
        elif not high:
            pattern = "low"
        elif all(abs(value) <= limit for value in tail):
            pattern = "settles"
        else:
            pattern = "stays high"
        finite = [value for value in values if math.isfinite(value)]
        finite_tail = [value for value in tail if math.isfinite(value)]
        summary.append({
            "phase": phase, "repeat": repeat, "module": module,
            "channel": "negative" if negative else "positive",
            "immediate_v": immediate, "first_fresh_v": values[0] if values else math.nan,
            "fresh_conversions": len(values), "above_limit": len(high),
            "last_above_index": fresh[high[-1]]["index"] if high else None,
            "last_above_s": fresh[high[-1]]["since_selection_s"] if high else None,
            "tail_median_v": statistics.median(finite_tail) if finite_tail else math.nan,
            "max_abs_v": max(map(abs, finite)) if finite else math.nan,
            "voltage_overflow_flags": sum(bool(row["v_overflow"]) for row in fresh),
            "timed_out": any(row["note"] for row in group),
            "pattern": pattern})
    return summary


def interpretation(summary):
    """Hints for the next discriminating step; only an independent measurement decides."""
    baseline = [entry for entry in summary if entry["phase"] == "baseline"]
    switched = [entry for entry in summary if entry["phase"] == "switch"]
    hints = []
    if any(entry["pattern"] != "low" for entry in baseline):
        hints.append("The untouched original channel already reads above the limit or gives no fresh conversion.")
    if switched and any(entry["pattern"] == "settles" for entry in switched) and all(
            entry["pattern"] in ("low", "settles") for entry in switched):
        hints.append("Every high reading after a channel selection decays below the limit: consistent with ADC "
                     "settling, which the runtime's check (first fresh conversion) would read as a voltage.")
    if any(entry["pattern"] == "stays high" for entry in switched):
        hints.append("At least one channel stays above the limit up to its last fresh conversion: a real voltage "
                     "or a channel without meaning for this module. Measure the output before any contact.")
    if any(entry["pattern"] == "no fresh conversion" for entry in summary):
        hints.append("A ready flag never cycled within CONVERSION_TIMEOUT_S on at least one channel.")
    if summary and all(entry["pattern"] == "low" for entry in summary):
        hints.append("Every channel read low here; compare with the runtime's discharge observations.")
    hints.append("This is not proof of discharge: only an independent HV measurement confirms or excludes a voltage.")
    return hints


def report(summary):
    print(f"\nHV ADC channels (|V| limit {LIMIT_V:g} V; fresh = ready flag seen clear, then set):")
    print(f"{'phase':9}{'rep':>4}  {'module':7}{'channel':9}{'immediate':>11}{'1st fresh':>11}{'last-5 med':>12}{'max |V|':>10}  pattern")
    for entry in summary:
        settled = (f" (last >{LIMIT_V:g} V at #{entry['last_above_index']}, {entry['last_above_s']:.2f} s)"
                   if entry["pattern"] == "settles" else "")
        print(f"{entry['phase']:9}{entry['repeat']:>4}  HV{entry['module']:<5}{entry['channel']:9}"
              f"{entry['immediate_v']:>11.3f}{entry['first_fresh_v']:>11.3f}{entry['tail_median_v']:>12.3f}"
              f"{entry['max_abs_v']:>10.3f}  {entry['pattern']}{settled}")
    for hint in interpretation(summary):
        print("- " + hint)


def save_plot(directory, rows, limit=LIMIT_V):
    try:
        from matplotlib.figure import Figure
    except ImportError:
        return None
    figure = Figure(figsize=(11, 4.5), constrained_layout=True)
    axes = figure.subplots(1, len(HV_MODULES), sharey=True)
    for axis, module in zip(axes, HV_MODULES):
        series = {}
        for row in rows:
            if row["module"] == module and row["voltage_valid"] is True and math.isfinite(row["voltage_v"]):
                series.setdefault((row["phase"], row["repeat"], row["negative_channel"]), []).append(row)
        for (phase, repeat, negative), points in series.items():
            channel = "neg" if negative else "pos"
            axis.plot([point["since_selection_s"] for point in points], [abs(point["voltage_v"]) for point in points],
                      marker=".", linestyle="-" if negative else "--",
                      label=f"{phase} {repeat} {channel}" if phase == "switch" else f"baseline {channel}")
        axis.axhline(limit, color="black", linewidth=.8, linestyle=":")
        axis.set_yscale("symlog", linthresh=.1)
        axis.set(title=f"HV{module}", xlabel="time since channel selection (s)")
        if series:
            axis.legend(fontsize=7)
    axes[0].set_ylabel("|ADC voltage| (V)")
    path = Path(directory) / "adc_probe.png"
    figure.savefig(path, dpi=150)
    try:
        from IPython.display import Image, display
        display(Image(filename=str(path)))
    except ImportError:
        pass
    return path


class Probe:
    """One probe run: claim, forced-OFF connect, ADC channel sequences, the runtime's verified shutdown."""
    def __init__(self, output_dir, esi_factory, *, guard, base, operator_restarts=(), trace_factory=None,
                 repeats=REPEATS, conversions=CONVERSIONS, baseline_conversions=BASELINE_CONVERSIONS,
                 conversion_timeout_s=CONVERSION_TIMEOUT_S, poll_s=POLL_S, clock=time.monotonic, sleep=time.sleep):
        self.output_dir = Path(output_dir)
        self.esi_factory, self.guard, self.base, self.trace_factory = esi_factory, guard, base, trace_factory
        self.repeats, self.conversions, self.baseline_conversions = repeats, conversions, baseline_conversions
        self.conversion_timeout_s, self.poll_s = conversion_timeout_s, poll_s
        self.clock, self.sleep = clock, sleep
        self.stop = threading.Event()
        self.rows, self.errors = [], []
        self.esi = self.backend = self.trace = self.directory = self.stream = None
        self.started = None
        self.esi_constructor_attempted = self.connect_attempted = self.connected = False
        self.hardware_started = self.shutdown_confirmed = False
        self.metadata = {"started_utc": utc(), "outcome": "running", "errors": self.errors,
                         "shutdown_confirmed": False, "operator_restarts": list(operator_restarts),
                         "esi_port": ESI_COM, "esi_baud": ESI_BAUD, "kind": "adc_probe",
                         "settings": {"repeats": repeats, "conversions": conversions,
                                      "baseline_conversions": baseline_conversions,
                                      "conversion_timeout_s": conversion_timeout_s, "poll_s": poll_s, "limit_v": LIMIT_V},
                         "flag_bits": {"v_ready": READY_V, "v_overflow": OVERFLOW_V,
                                       "i_ready": READY_I, "i_overflow": OVERFLOW_I},
                         "scope": "ADC channel selection and reads only; no output, gate, target or heater enabled. "
                                  "Readings describe the ADC, not a certified discharge.",
                         "probe_complete": False}

    def validate(self):
        for name, value, minimum in (("repeats", self.repeats, 1), ("conversions", self.conversions, 6),
                                     ("baseline_conversions", self.baseline_conversions, 1)):
            if isinstance(value, bool) or not isinstance(value, int) or value < minimum:
                raise ValueError(f"{name} must be an integer >= {minimum}")
        for name, value in (("conversion_timeout_s", self.conversion_timeout_s), ("poll_s", self.poll_s)):
            if isinstance(value, bool) or not isinstance(value, (int, float)) or not math.isfinite(value) or value <= 0:
                raise ValueError(f"{name} must be finite and positive")

    def fault(self, error):
        self.errors.append(f"{type(error).__name__}: {error}")

    def claim(self):
        if _ADC_GUARD.get("probe") is not None or _ADC_GUARD.get("guard") not in (None, self.guard):
            raise RuntimeError("A previous probe in this kernel still owns an uncertain transport; do not rerun.")
        self.guard.check()
        self.output_dir.mkdir(parents=True, exist_ok=True)
        sync_directory(self.output_dir.parent)
        _ADC_GUARD.update(probe=self, guard=self.guard)
        try:
            self.directory = self.output_dir / (datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ_") + uuid4().hex[:8])
            self.directory.mkdir()
            sync_directory(self.output_dir)
            self.metadata["directory"] = str(self.directory)
            save_json(self.directory / "metadata.json", self.metadata)
            self.stream = (self.directory / "samples.csv").open("x", newline="", encoding="utf-8")
            self.writer = csv.DictWriter(self.stream, fieldnames=FIELDS)
            self.writer.writeheader()
            self.stream.flush()
            os.fsync(self.stream.fileno())
            sync_directory(self.directory)
            self.metadata["experiment_claim"] = self.guard.claim(self.directory, self.directory / "metadata.json")
            save_json(self.directory / "metadata.json", self.metadata)
        except BaseException as error:
            # Storage preflight failed before constructing/opening the instrument.
            self.fault(error)
            self.metadata.update(outcome="interrupted" if isinstance(error, KeyboardInterrupt) else "error",
                                 ended_utc=utc(), esi_instance_created=False, shutdown_confirmed=False)
            try:
                if self.stream is not None:
                    self.stream.close()
                if self.directory is not None and self.directory.is_dir():
                    save_json(self.directory / "metadata.json", self.metadata)
            finally:
                if self.guard.abandon_before_hardware() is True:
                    _ADC_GUARD.clear()
            raise

    def append(self, row):
        self.writer.writerow(row)
        self.stream.flush()  # Each conversion survives a later exception/interrupt.
        self.rows.append(row)

    def poison_state(self):
        try:
            state = (self.backend if self.backend is not None else self.esi)._transport_poisoned
            return state if type(state) is bool else None
        except BaseException:
            return None

    def native(self, function, *args):
        self.backend._raise_if_transport_poisoned()
        result = function(self.backend, *args)
        self.backend._raise_if_transport_poisoned()
        status = result[0] if isinstance(result, tuple) else result
        self.backend._raise_on_status(status, f"{function.__name__}{args}")
        return result[1:] if isinstance(result, tuple) else ()

    def locked(self, label, function, budget_s):
        # The runtime's own lock and timeout: a hung DLL call poisons the transport, never a second caller.
        return self.backend._call_locked_with_timeout(function, budget_s, "adc_probe_" + label)

    def wait_fresh(self, address):
        """Same handshake as the runtime's discharge check: each ready bit seen clear, then set."""
        armed = 0
        deadline = self.clock() + self.conversion_timeout_s
        while not self.stop.is_set():
            flags, = self.native(self.base.get_module_data_ready_flags, address)
            armed |= ~flags & READY
            if flags & armed & READY == READY:
                return flags
            # Drain a value whose ready bit was never seen clear: it may predate the last read.
            if flags & READY_V and not armed & READY_V:
                self.native(self.base.get_hv_supply_output_voltage, address)
            if flags & READY_I and not armed & READY_I:
                self.native(self.base.get_hv_supply_output_current, address)
            if self.clock() >= deadline:
                return None
            self.sleep(self.poll_s)
        return None

    def sequence(self, phase, repeat, address, negative, high_current, count, *, select):
        rows = []

        def measure():
            if select:
                self.native(self.base.set_hv_supply_meas_ranges, address, negative, high_current)
                if tuple(self.native(self.base.get_hv_supply_meas_ranges, address)) != (negative, high_current):
                    raise RuntimeError(f"HV{address} ADC channel selection failed")
            selected = self.clock()

            def record(index, fresh, flags, note=""):
                valid_v, voltage = self.native(self.base.get_hv_supply_output_voltage, address)
                valid_i, current = self.native(self.base.get_hv_supply_output_current, address)
                now = self.clock()
                rows.append({"phase": phase, "repeat": repeat, "module": address, "negative_channel": negative,
                             "high_current_range": high_current, "index": index, "fresh": fresh,
                             "since_selection_s": now - selected, "elapsed_s": now - self.started, "utc": utc(),
                             "flags": flags, "v_ready": bool(flags & READY_V), "v_overflow": bool(flags & OVERFLOW_V),
                             "i_ready": bool(flags & READY_I), "i_overflow": bool(flags & OVERFLOW_I),
                             "voltage_valid": valid_v, "voltage_v": voltage, "current_valid": valid_i,
                             "current_a": current, "note": note})

            # Index 0: the value available at once, as a routine read or the runtime's drain read sees it.
            flags, = self.native(self.base.get_module_data_ready_flags, address)
            record(0, False, flags)
            for index in range(1, count + 1):
                flags = self.wait_fresh(address)
                if flags is None:
                    if not self.stop.is_set():
                        rows.append({**rows[-1], "index": index, "fresh": False,
                                     "since_selection_s": self.clock() - selected, "elapsed_s": self.clock() - self.started,
                                     "utc": utc(), "voltage_valid": False, "voltage_v": math.nan,
                                     "current_valid": False, "current_a": math.nan,
                                     "note": f"no fresh conversion within {self.conversion_timeout_s:g} s"})
                    return
                record(index, True, flags)

        channel = "negative" if negative else "positive"
        try:
            self.locked(f"{phase}{repeat}_hv{address}_{channel}", measure,
                        (count + 1) * (self.conversion_timeout_s + 2.) + 10.)
        finally:
            for row in list(rows):  # Disk I/O after the native sequence, never on the DLL thread.
                self.append(row)
        fresh = [row["voltage_v"] for row in rows if row["fresh"]]
        if fresh:
            print(f"{phase} {repeat} HV{address} {channel}: immediate {rows[0]['voltage_v']:.3f} V, "
                  f"first fresh {fresh[0]:.3f} V, last {fresh[-1]:.3f} V ({len(fresh)} fresh conversions)")

    def probe(self):
        saved = {}

        def remember():
            for address in HV_MODULES:
                saved[address] = tuple(self.native(self.base.get_hv_supply_meas_ranges, address))

        self.locked("read_channels", remember, 10.)
        self.metadata["original_channels"] = {address: {"negative": negative, "high_current": high}
                                              for address, (negative, high) in saved.items()}
        try:
            for address in HV_MODULES:
                if self.stop.is_set():
                    return
                self.sequence("baseline", 0, address, *saved[address], self.baseline_conversions, select=False)
            for repeat in range(1, self.repeats + 1):
                for negative in (False, True):
                    for address in HV_MODULES:
                        if self.stop.is_set():
                            return
                        self.sequence("switch", repeat, address, negative, saved[address][1], self.conversions, select=True)
            self.metadata["probe_complete"] = True
        finally:
            if saved and self.poison_state() is False:
                def restore():
                    for address, (negative, high) in saved.items():
                        self.native(self.base.set_hv_supply_meas_ranges, address, negative, high)
                        if tuple(self.native(self.base.get_hv_supply_meas_ranges, address)) != (negative, high):
                            raise RuntimeError(f"HV{address} ADC channel restore failed")
                try:
                    self.locked("restore_channels", restore, 10.)
                    self.metadata["channels_restored"] = True
                except BaseException as error:
                    self.fault(error)
                    self.metadata["channels_restored"] = False

    def hardware(self):
        self.esi_constructor_attempted = True
        self.esi = self.esi_factory(self.directory)
        self.backend = inline_backend(self.esi, self.base)
        if self.trace_factory is not None:
            self.trace = self.trace_factory(self.directory)
            self.trace.install(self.base, self.backend)
        self.connect_attempted = True
        if self.esi.connect() is not True:
            raise RuntimeError("ESI connect unconfirmed")
        self.connected = True
        diagnostic = self.esi.collect_diagnostics()
        self.metadata["initial_diagnostics"] = diagnostic
        problem = hv_off_problem(diagnostic)
        if problem:
            raise RuntimeError(problem)
        print("Connected; the runtime forced HV and heater OFF. Recording the HV ADC channels ...")
        self.probe()

    def discharge_observation(self, state):
        # Keep every round of the runtime's own discharge check, not only the last.
        observations = self.metadata.setdefault("discharge_observations", [])
        if len(observations) < 2000:
            observations.append({"elapsed_s": self.clock() - self.started, **state})
        self.metadata["last_discharge"] = state

    def shutdown(self):
        if self.esi is None:
            self.shutdown_confirmed = not self.esi_constructor_attempted
            return
        try:
            if self.poison_state() is not False:
                self.fault(RuntimeError("ESI transport poisoned or unknown: no further DLL call, OFF or Close attempted"))
                return
            print("Verified shutdown: runtime OFF, then its own HV discharge check (up to 60 s) ...")
            try:
                closed = self.esi.disconnect(on_discharge=self.discharge_observation)
                # Before connect() no port was opened; afterwards only a real connection can be confirmed.
                self.shutdown_confirmed = closed is True and (self.connected or not self.connect_attempted)
                if not self.shutdown_confirmed:
                    raise RuntimeError("ESI OFF/discharge/port closure not confirmed")
            except BaseException as error:
                self.fault(error)
        finally:
            if self.trace is not None:
                self.metadata["native_trace"] = self.trace.finish(self.poison_state() is False)

    def run(self):
        self.validate()
        self.started = self.clock()
        self.claim()  # The CSV, metadata and persistent guard exist before hardware access.
        try:
            self.hardware_started = True
            self.guard.mark_hardware_started()
        except BaseException as error:
            self.fault(error)
            self.metadata.update(outcome="error", ended_utc=utc())
            save_json(self.directory / "metadata.json", self.metadata)
            raise
        done = threading.Event()

        def owner():
            # The only thread that calls the instrument; Stop is a request it honours between conversions.
            try:
                self.hardware()
                self.metadata["outcome"] = "stopped" if self.stop.is_set() else "complete"
            except BaseException as error:
                self.fault(error)
                self.metadata["outcome"] = "error"
            finally:
                try:
                    self.shutdown()
                except BaseException as error:
                    self.fault(error)
                finally:
                    done.set()

        threading.Thread(target=owner, name="esi-adc-probe", daemon=True).start()
        while not done.is_set():
            try:
                done.wait(.2)
            except KeyboardInterrupt:
                if self.stop.is_set():
                    print("Stop already requested: waiting for the verified shutdown. Do not restart the kernel.")
                else:
                    print("Stop requested: finishing the current conversion, restoring the ADC channels, then verified shutdown.")
                self.stop.set()
        return self.finish()

    def finish(self):
        poisoned = self.poison_state() if self.esi is not None else False
        try:
            self.stream.close()
        except BaseException as error:
            self.fault(error)
        if self.errors and self.metadata["outcome"] == "complete":
            self.metadata["outcome"] = "error"
        if not self.shutdown_confirmed:
            self.metadata["outcome"] = "shutdown_unconfirmed"
        summary = summarize(self.rows)
        self.metadata.update(ended_utc=utc(), samples=len(self.rows), summary=summary,
                             hints=interpretation(summary), esi_instance_created=self.esi is not None,
                             shutdown_confirmed=self.shutdown_confirmed, esi_transport_poisoned=poisoned)
        save_json(self.directory / "metadata.json", self.metadata)
        try:
            cleared = self.guard.finalize(shutdown_confirmed=self.shutdown_confirmed, owners_idle=True,
                                          auxiliary_closed=True, transport_poisoned=poisoned)
        except BaseException as error:
            self.fault(error)
            cleared = False
            self.metadata.update(outcome="shutdown_unconfirmed", guard_release_error=f"{type(error).__name__}: {error}")
            save_json(self.directory / "metadata.json", self.metadata)
        if cleared is True:
            _ADC_GUARD.clear()
        report(summary)
        try:
            save_plot(self.directory, self.rows)
        except BaseException as error:
            print(f"Plot not saved ({type(error).__name__}: {error}); the CSV and metadata are complete.")
        if not self.shutdown_confirmed:
            print("\nSHUTDOWN UNCONFIRMED: the runtime's HV discharge check did not pass (see discharge_observations). "
                  "Treat HV as live until measured. The COM guard is retained; the next run asks for the RESTART declaration.")
        if self.errors:
            print("\n".join(self.errors))
        print(f"{self.metadata['outcome']}: {len(self.rows)} conversions"
              f"{' (probe complete)' if self.metadata['probe_complete'] else ''} → {self.directory}")
        return self


def main():
    if _ADC_GUARD:
        raise RuntimeError("Unfinished probe in this kernel. Make the instrument safe, terminate this kernel, then use operator recovery in a fresh kernel.")
    if sys.platform != "win32":
        raise RuntimeError("ESI requires Windows and the bundled vendor DLL. No hardware accessed.")
    if isinstance(ESI_COM, bool) or not isinstance(ESI_COM, int) or not 1 <= ESI_COM <= 255:
        raise ValueError("ESI_COM must be a COM number from 1 to 255")
    if ESI_BAUD != 230400:
        raise ValueError("ESI_BAUD must be 230400; no baud fallback is attempted")
    root = find_plugins(PLUGINS_DIR)
    fingerprints = runtime_fingerprints(root)
    guards = load_private(root / "esi/_experiment_guard.py")
    guard = guards.ExperimentGuard(plugin_dir=root / "esi", com=ESI_COM, kind="adc_probe",
                                   output_dir=OUTPUT_DIR, kernel_globals=globals(),
                                   legacy_pt_dirs=(NOTEBOOK_DIR / "pressure_temperature_runs",))
    probe = None
    try:
        records = guard.authorize_restart()
        _ADC_GUARD["guard"] = guard
        runtime = load_private(root / "esi/vendor/runtime/__init__.py", package=True, fresh=True)
        require_discharge_runtime(runtime)
        probe = Probe(OUTPUT_DIR,
                      lambda directory: runtime.ESI("adc-probe", ESI_COM, baudrate=ESI_BAUD,
                                                    log_dir=directory, process_backend=False),
                      guard=guard, base=sys.modules[runtime.__name__ + ".esi.esi_base"].ESIBase,
                      operator_restarts=records, trace_factory=NativeTrace)
        probe.metadata["runtime_sources"] = fingerprints
        return probe.run()
    except BaseException:
        if probe is None or (not probe.hardware_started and _ADC_GUARD.get("guard") is guard):
            if guard.abandon_before_hardware() is True:
                if _ADC_GUARD.get("guard") is guard:
                    _ADC_GUARD.clear()
            else:
                _ADC_GUARD["guard"] = guard
        raise


main()